> 在线讲义：[第 8 章 · 两种梯度：「试了打分」的似然比梯度 vs 「顺着公式求导」的解析梯度](https://wikieden.github.io/robot-motion-rl/lessons/08-two-gradients.html) · 本书《机器人运控与强化学习》

# 第 8 章 · 两种梯度：「试了打分」的似然比梯度 vs 「顺着公式求导」的解析梯度

配套讲义：仓库里 `robot-motion-rl/notes/lessons/08-two-gradients.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（玩具回报和样本是编的，重复实验用固定随机种子 0）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 真值 | −2(θ − 2) | `dJ_da(0.0)` | 4.0 |
| 第二步 似然比 | mean[(J − b)ε/σ] | `score_estimate(0, 0.5, eps)` | 2.999 |
| 第三步 解析 | mean[J'(a)] | `pathwise_estimate(0, 0.5, eps)` | 3.700 |
| 第四步 散布 | 重复 2000 次 | `spread(..., n)` | N=4：2.99 ± 2.46 vs 4.00 ± 0.51 |

例子（编的）：回报 J(a) = −(a − 2)²，策略 a = θ + σ·ε，ε ~ N(0, 1)，在 θ = 0、σ = 0.5 处求 dE[J]/dθ。真值 = 4。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `J`**：回报：a 越接近 2 越好。

In [ ]:
def J(a):
    """回报：a 越接近 2 越好。"""
    return -((a - 2.0) ** 2)

**函数 `dJ_da`**：回报对动作的导数（解析梯度要用到）：−2(a − 2)。

In [ ]:
def dJ_da(a):
    """回报对动作的导数（解析梯度要用到）：−2(a − 2)。"""
    return -2.0 * (a - 2.0)

**函数 `score_estimate`**：似然比估计：mean[(J(a) − b) · ε / σ]，b 取这批 J 的平均。

In [ ]:
def score_estimate(theta, sigma, eps, baseline=True):
    """似然比估计：mean[(J(a) − b) · ε / σ]，b 取这批 J 的平均。"""
    a = theta + sigma * eps
    j = J(a)
    b = j.mean() if baseline else 0.0
    return float(np.mean((j - b) * eps / sigma))

**函数 `pathwise_estimate`**：解析梯度：a = θ + σε 对 θ 的导数是 1，所以 dJ/dθ = mean[dJ/da(a)]。

In [ ]:
def pathwise_estimate(theta, sigma, eps):
    """解析梯度：a = θ + σε 对 θ 的导数是 1，所以 dJ/dθ = mean[dJ/da(a)]。"""
    return float(np.mean(dJ_da(theta + sigma * eps)))

**函数 `spread`**：重复估计 repeats 次（每次 n 个样本），返回 (平均, 标准差)。

In [ ]:
def spread(estimator, n, repeats=2000, seed=0, **kw):
    """重复估计 repeats 次（每次 n 个样本），返回 (平均, 标准差)。"""
    rng = np.random.default_rng(seed)
    vals = [estimator(0.0, 0.5, rng.standard_normal(n), **kw) for _ in range(repeats)]
    return float(np.mean(vals)), float(np.std(vals))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 真梯度")
print("dE[J]/dθ 在 θ=0 = −2(0 − 2) =", dJ_da(0.0))

In [ ]:
print("\n第二步 一次估计（4 个样本）")
eps = np.array([0.5, -1.0, 1.5, 0.2])
print(f"似然比 = {score_estimate(0.0, 0.5, eps):.3f}；解析 = {pathwise_estimate(0.0, 0.5, eps):.3f}")

In [ ]:
print("\n第三步 重复 2000 次看散布")
for n in (4, 64):
    ms, ss = spread(score_estimate, n)
    mp, sp = spread(pathwise_estimate, n)
    print(f"N={n}: 似然比 {ms:.2f} ± {ss:.2f}；解析 {mp:.2f} ± {sp:.2f}")

## 3. 代码里才看得到的

- 似然比估计在 N = 4 时平均只有 2.99，不是 4：baseline 用了**同一批样本自己的平均**，会带来 (N − 1)/N 的缩小（4 × 3/4 = 3）。N = 64 时这个偏差变成 63/64，平均回到 3.95。实际实现里 baseline 用 critic 或「留一法」平均，就没有这个偏差。
- 解析梯度的散布只来自 ε 本身（J' 是 a 的一次函数），所以 N = 4 时标准差 = 2σ/√4 = 0.5，和跑出来的 0.51 一致。

## 4. 自己改一改

- 把 baseline 设成 0（`baseline=False`），看似然比的散布变大多少。
- 把 σ 从 0.5 改成 0.1，两种估计的散布分别怎么变？（提示：似然比里有 1/σ。）

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 调糖两办法：冲几杯尝分数，或按公式求导数；
> 似然比只看分，黑箱能用，方差大，靠样本压；
> 解析沿路求导，每个样本带方向，要环境可微；
> 同一模型比一比，解析省六十四倍。